# **데이터 전처리**


---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록  ───────────────────────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)



---



## ⏺️ **학습용 데이터의 이해**

### **[예제] 학생 성적 데이터 전처리 파이프라인**

In [ ]:
# [실습] 학생 성적 데이터 전처리 파이프라인 — Raw → 정제 → 분리 → 전처리 → ML  (2차시 학습용 데이터와 전처리)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1. 문제가 있는 학습 데이터 만들기 (일부러 NaN, 100시간, None 을 넣음)
data = {
    "study_hours": [1, 2, 3, 4, np.nan, 6, 7, 8, 2, 3, 4, 5, 6, 7, 100, 4],
    "attendance":  [55, 60, 65, 70, 75, 80, 85, 95, 58, np.nan, 72, 76, 82, 88, 90, 70],
    "assignment":  [40, 50, 55, 60, 65, 75, 80, 95, 45, 55, 68, 72, 82, 87, 90, 60],
    "major": ["CS", "CS", "AI", "SW", "AI", "CS", "SW", "AI", "CS", None, "SW", "AI", "CS", "SW", "AI", "SW"],
    "pass":  [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0],
}
df = pd.DataFrame(data)

# 2. 데이터 상태 확인 (EDA)
print(df.head()); print("\n크기:", df.shape)
print("\n결측치\n", df.isna().sum())
print("\n기초 통계\n", df.describe())          # study_hours 최대값 100 → 이상!

# 3. 잘못된 값 처리 — 도메인 지식: 하루 공부시간은 0~24시간
df.loc[(df["study_hours"] < 0) | (df["study_hours"] > 24), "study_hours"] = np.nan

# 4. 중복 제거
print("\n중복 개수:", df.duplicated().sum())
df = df.drop_duplicates()

# 5. X(문제) / y(정답) 분리
X = df.drop(columns=["pass"]); y = df["pass"]

# 6. Train / Test 분리 — 전처리 fit 전에 먼저 나눈다 (Data Leakage 방지)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("\nTrain:", X_train.shape, " Test:", X_test.shape)

# 7. 수치형: 결측 → 중앙값, 그 다음 표준화
numeric_features = ["study_hours", "attendance", "assignment"]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])

# 8. 범주형: 결측 → 최빈값, 그 다음 One-Hot
categorical_features = ["major"]
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

# 9. ColumnTransformer — 컬럼 종류별로 다른 전처리
preprocessor = ColumnTransformer([("num", numeric_pipeline, numeric_features),
                                  ("cat", categorical_pipeline, categorical_features)])

# 10. 전처리 + 모델을 하나의 Pipeline 으로
model = Pipeline([("preprocessing", preprocessor),
                  ("classifier", LogisticRegression(max_iter=1000))])

# 11. 학습: fit 은 Train 에만 — 전처리 통계(중앙값·평균·표준편차)도 Train 에서만 계산됨
model.fit(X_train, y_train)

# 12. 평가: Test 는 transform 만 거친다
print("\nAccuracy:", model.score(X_test, y_test))

# 13. 전처리된 데이터를 직접 보기 — 모델이 보는 것은 결국 숫자 행렬
processor = model.named_steps["preprocessing"]
processed_df = pd.DataFrame(processor.transform(X_train), columns=processor.get_feature_names_out())
print("\n전처리 결과\n", processed_df.head().round(2))




---



## ⏺️ **데이터 전처리**



---



### [예제] 데이터 전처리

 `ml_preprocessing_practice_dataset.csv`를 사용하여 다음 내용을 실습합니다.

1. 데이터 구조 확인
2. 결측치 처리
3. 중복 데이터 제거
4. 잘못된 값 처리
5. 이상치(IQR) 탐지
6. 범주형 데이터 인코딩
7. Scaling
8. Feature Engineering
9. Train / Test 분리
10. 데이터 누수(Data Leakage)를 막는 Pipeline

#### **1. 데이터 불러오기**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# 데이터 불러오기
file_path = "ml_preprocessing_practice_dataset.csv"
df = pd.read_csv(file_path)

print("데이터 크기:", df.shape)
display(df.head(10))

#### **2. 데이터 확인**

In [ ]:
print("=== 데이터 정보 ===")
df.info()

print("\n=== 결측치 개수 ===")
display(df.isna().sum())

print("\n=== 기초 통계량 ===")
display(df.describe(include="all"))

In [ ]:
print("city 고유값:")
print(df["city"].unique())

#### **3-4. 중복 데이터와 잘못된 값**
전처리에서는 통계보다 먼저 **도메인 규칙**으로 오류를 찾을 수 있습니다.

- 나이 = -5 → 불가능
- 출석률 = 150% → 불가능
- 키 = 190cm → 드물지만 가능


- **중복된 값**

In [ ]:
print("전체 중복 행 개수:", df.duplicated().sum())

duplicates = df[df.duplicated(keep=False)]
display(duplicates)

In [ ]:
id_duplicates = df[df.duplicated(subset=["student_id"], keep=False)]

print("student_id 중복 후보 개수:", len(id_duplicates))
display(id_duplicates.sort_values("student_id"))

In [ ]:
df = df.drop_duplicates().copy()
print("완전 중복 제거 후 데이터 크기:", df.shape)

- **잘못된 값 탐지** 및 NaN으로 변환

In [ ]:
print("잘못된 age")
display(df[df["age"] < 0])

print("잘못된 attendance")
display(df[(df["attendance"] < 0) | (df["attendance"] > 100)])

In [ ]:
df.loc[df["age"] < 0, "age"] = np.nan

df.loc[
    (df["attendance"] < 0) | (df["attendance"] > 100),
    "attendance"
] = np.nan

display(df.isna().sum())

- **결측치 처리**

| 컬럼 | 방법 | 이유 |
|---|---|---|
| age | 평균 | 연속형 수치 |
| income | 중앙값 | 이상치 영향 감소 |
| city | 최빈값 | 범주형 |
| temp | 보간 | 시간 흐름이 있는 연속형 |
| attendance | 중앙값 | 범위 오류를 NaN으로 바꾼 뒤 보완 |

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df["age"] = df["age"].fillna(df["age"].mean())
df["income"] = df["income"].fillna(df["income"].median())
df["city"] = df["city"].fillna(df["city"].mode()[0])
df["temp"] = df["temp"].interpolate()
df["attendance"] = df["attendance"].fillna(df["attendance"].median())

display(df.isna().sum())

- **범주 표현 불일치 통일**

In [ ]:
city_map = {
    "서울": "서울", "Seoul": "서울", "서울시": "서울",
    "부산": "부산", "Busan": "부산", "부산시": "부산",
    "대전": "대전", "Daejeon": "대전", "대전시": "대전",
    "인천": "인천", "Incheon": "인천", "인천시": "인천",
    "광주": "광주", "Gwangju": "광주", "광주시": "광주",
}

df["city"] = df["city"].replace(city_map)
print(df["city"].value_counts())

#### **5. 이상치(Outlier)** — IQR로 찾되 무조건 삭제하지 않는다

\[
IQR = Q_3 - Q_1
\]

일반적인 탐지 기준:

\[
x < Q_1 - 1.5 	imes IQR
\]

또는

\[
x > Q_3 + 1.5 	imes IQR
\]

In [ ]:
Q1 = df["height"].quantile(0.25)
Q3 = df["height"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = {IQR:.2f}")
print(f"정상 범위 = {lower:.2f} ~ {upper:.2f}")

outliers = df[(df["height"] < lower) | (df["height"] > upper)]
display(outliers[["student_id", "height"]])

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(df["height"].dropna(), vert=False)
plt.title("Height Boxplot")
plt.xlabel("height (cm)")
plt.show()

- **이상치 처리 예시: clip()**

이상치를 자동 삭제하지 않고, 필요하면 경계값으로 제한할 수 있습니다.

In [ ]:
df["height_clipped"] = df["height"].clip(lower, upper)

display(
    df.loc[
        df["height"] != df["height_clipped"],
        ["student_id", "height", "height_clipped"]
    ]
)

#### **6. 범주형 데이터 Encoding**

`city`는 순서가 없는 명목형 범주이므로 One-Hot Encoding을 적용합니다.

In [ ]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

city_encoded = encoder.fit_transform(df[["city"]])

city_encoded_df = pd.DataFrame(
    city_encoded,
    columns=encoder.get_feature_names_out(["city"])
)

display(city_encoded_df.head())

#### **7. Scaling** — 크기가 다른 Feature를 같은 눈금으로

거리 기반 모델(kNN, SVM)과 경사하강법 기반 모델은 Feature 크기에 영향을 받을 수 있습니다.

In [ ]:
scale_cols = ["age", "income", "temp", "attendance", "height_clipped"]

scaler = StandardScaler()
scaled = scaler.fit_transform(df[scale_cols])

scaled_df = pd.DataFrame(
    scaled,
    columns=[f"{c}_scaled" for c in scale_cols]
)

display(scaled_df.head())

- **StandardScaler / MinMaxScaler / RobustScaler 비교**

In [ ]:
sample = df[["income"]]

standard = StandardScaler().fit_transform(sample)
minmax = MinMaxScaler().fit_transform(sample)
robust = RobustScaler().fit_transform(sample)

compare = pd.DataFrame({
    "original_income": df["income"],
    "StandardScaler": standard.ravel(),
    "MinMaxScaler": minmax.ravel(),
    "RobustScaler": robust.ravel()
})

display(compare.head(10))

#### **8. Feature Engineering**

기존 열을 이용해 새로운 의미 있는 Feature를 만듭니다.

In [ ]:
df["age_from_birth"] = 2026 - df["birth_year"]
df["total_price"] = df["price"] * df["quantity"]
df["weekday"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month

display(
    df[
        ["birth_year", "age_from_birth",
         "price", "quantity", "total_price",
         "date", "weekday", "month"]
    ].head()
)

#### **9. Train / Test 분**리

`purchased`를 분류 Target으로 사용합니다.

- Train: 80%
- Test: 20%
- `random_state=42`: 재현성
- `stratify=y`: 클래스 비율 유지

In [ ]:
features = [
    "age", "income", "city", "temp",
    "attendance", "height_clipped",
    "total_price", "weekday", "month"
]

X = df[features]
y = df["purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
print("전체 클래스 비율")
display(y.value_counts(normalize=True))

print("Train 클래스 비율")
display(y_train.value_counts(normalize=True))

print("Test 클래스 비율")
display(y_test.value_counts(normalize=True))

#### 중요: 데이터 누수(Data Leakage) 방지

전처리 파라미터를 전체 데이터에서 먼저 계산하면 Test 정보가 Train에 섞일 수 있습니다.

잘못된 순서:

```python
X_scaled = scaler.fit_transform(X)
train_test_split(X_scaled)
```

권장 순서:

```text
전체 데이터
   ↓
Train / Test 분리
   ↓
Train에만 fit
   ↓
Test에는 transform만
```

### [종합] 전처리 Pipeline

In [ ]:
numeric_features = [
    "age", "income", "temp", "attendance",
    "height_clipped", "total_price",
    "weekday", "month"
]

categorical_features = ["city"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("전처리 후 Train shape:", X_train_processed.shape)
print("전처리 후 Test shape :", X_test_processed.shape)

- **전처리 결과의 Feature 이름 확인**

In [ ]:
feature_names = preprocessor.get_feature_names_out()

processed_train = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

display(processed_train.head())

### [정리]

```text
데이터 확인
→ 중복 제거
→ 잘못된 값 탐지
→ 결측치 처리
→ 이상치 판단
→ 범주형 인코딩
→ Scaling
→ Feature Engineering
→ Train/Test 분리
→ Pipeline으로 데이터 누수 방지
```

> **통계적으로 이상하다고 해서 틀린 값은 아니다. 전처리는 통계 + 도메인 지식을 함께 사용해야 한다.**

## ⏺️ **데이터 전처리 파이프라인**

### **[실습] 학생 성적 데이터 전처리 파이프라인**

- Raw → 정제 → 분리 → 전처리 → ML

#### **1. 문제가 있는 학습 데이터 만들기** (일부러 NaN, 100시간, None 을 넣음)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
data = {
    "study_hours": [1, 2, 3, 4, np.nan, 6, 7, 8, 2, 3, 4, 5, 6, 7, 100, 4],
    "attendance":  [55, 60, 65, 70, 75, 80, 85, 95, 58, np.nan, 72, 76, 82, 88, 90, 70],
    "assignment":  [40, 50, 55, 60, 65, 75, 80, 95, 45, 55, 68, 72, 82, 87, 90, 60],
    "major": ["CS", "CS", "AI", "SW", "AI", "CS", "SW", "AI", "CS", None, "SW", "AI", "CS", "SW", "AI", "SW"],
    "pass":  [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0],
}
df = pd.DataFrame(data)

#### **2. 데이터 상태 확인 (EDA)**

In [ ]:
print(df.head()); print("\n크기:", df.shape)
print("\n결측치\n", df.isna().sum())
print("\n기초 통계\n", df.describe())          # study_hours 최대값 100 → 이상!


#### **3. 잘못된 값 처리** — 도메인 지식: 하루 공부시간은 0~24시간

In [ ]:
df.loc[(df["study_hours"] < 0) | (df["study_hours"] > 24),
       "study_hours"
] = np.nan


#### **4. 중복 제거**

In [ ]:
print("\n중복 개수:", df.duplicated().sum())
df = df.drop_duplicates()

#### **5. X(문제) / y(정답) 분리**

In [ ]:
X = df.drop(columns=["pass"])
y = df["pass"]

#### **6. Train / Test 분리** — 전처리 fit 전에 먼저 나눈다 (Data Leakage 방지)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    random_state=42,
                                                    stratify=y)
print("\nTrain:", X_train.shape, " Test:", X_test.shape)


#### **7. 수치형: 결측 → 중앙값, 그 다음 표준화**

In [ ]:
numeric_features = ["study_hours", "attendance", "assignment"]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())])

#### **8. 범주형: 결측 → 최빈값, 그 다음 One-Hot**

In [ ]:
categorical_features = ["major"]
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

#### **9. ColumnTransformer — 컬럼 종류별로 다른 전처리**

In [ ]:
preprocessor = ColumnTransformer([("num", numeric_pipeline, numeric_features),
                                  ("cat", categorical_pipeline, categorical_features)])


#### **10. 전처리 + 모델을 하나의 Pipeline 으로**

In [ ]:
model = Pipeline([("preprocessing", preprocessor),
                  ("classifier", LogisticRegression(max_iter=1000))])


#### **11. 학습**: fit 은 Train 에만 — 전처리 통계(중앙값·평균·표준편차)도 Train 에서만 계산됨

In [ ]:
model.fit(X_train, y_train)

#### **12. 평가**: Test 는 transform 만 거친다

In [ ]:
print("\nAccuracy:", model.score(X_test, y_test))

#### **13. 전처리된 데이터를 직접 보기** — 모델이 보는 것은 결국 숫자 행렬

In [ ]:
processor = model.named_steps["preprocessing"]
processed_df = pd.DataFrame(processor.transform(X_train), columns=processor.get_feature_names_out())
print("\n전처리 결과\n", processed_df.head().round(2))



---

